In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import copy
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

In [ ]:
# Set a fixed seed, ensuring random operations produce the same results
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Choose the device which is used, using the GPU, if it is available
# Otherwise, use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Display the device which is used
print("Device:", device)

In [ ]:
# State the path of the input directory, where the dataset is stored
KAGGLE_INPUT = Path("/kaggle/input")
# Find each of the relevant files
csv_candidates = list(KAGGLE_INPUT.rglob("*.csv"))
image_candidates = list(KAGGLE_INPUT.rglob("*.jpg"))
# Introduce the list of files which are found, and list the files
print("CSV files found:")
for x in csv_candidates:
    print(" ", x)
# State how many images are found
print("\nNumber of JPG images found:", len(image_candidates))

In [ ]:
# Select csv files with 'ground', 'truth', or '2019' in their file name
ground_truth_candidates = [path for path in csv_candidates if "ground" in path.name.lower() or "truth" in path.name.lower() or "2019" in path.name.lower()]
# Introduce the files which are found 
print("Possible ground-truth files:")
# Loop through each of the files, displaying their paths
for path in ground_truth_candidates:
    print(" ", path)

In [ ]:
# Check whether the files are found 
if len(ground_truth_candidates) == 0:
    # Stop the program and display an error message, if none are found 
    raise FileNotFoundError("Could not find the ISIC 2019 ground-truth CSV files.")
# Select the first file from the list
ground_truth_path = ground_truth_candidates[0]
# Read file into a dataframe
df = pd.read_csv(ground_truth_path)
# Display the path of the file 
print("Ground truth file:")
print(ground_truth_path)
# Display the dimensions of the dataframe 
print("\nShape:", df.shape)
# Display the names of the columns
print("\nColumns:")
print(df.columns.tolist())
# Show the first five rows of the dataframe
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# Check whether any of the labels contain 'MEL', short for melanoma
if "MEL" not in df.columns:
    # Stop the program and display an error message, if it is not found
    raise ValueError("The ground-truth CSV does not contain a 'MEL' column.")
# Create a new column within the dataframe, assigning an integer
df["label"] = df["MEL"].astype(int)
# Count the number of times in which melanoma and non-melanoma occur
print(df["label"].value_counts())
# Melanoma is 1, while non-melanoma is false, displaying the count for each 
print("Melanoma:", int(df["label"].sum()), "| Non-melanoma:", int((df["label"] == 0).sum()))

In [ ]:
# Find each of the images from within the directory and sub-directories
all_images = list(KAGGLE_INPUT.rglob("*.jpg"))
# Create a dictionary, mapping the name of each file to the full path
image_lookup = {image.stem: str(image) for image in all_images}
# State the total number of unique images which are indexed
print("Images indexed:", len(image_lookup))

In [ ]:
# Define the name of the column which contains the images
image_column_id = "image"
# Check whether the expected column exists within the dataframe
if image_column_id not in df.columns:
    # If it is not present, display an error message, saying it cannot be found and showing the columns which are available 
    raise ValueError(f"Could not find the '{image_column_id}' column. " f"Available columns: {df.columns.tolist()}")
# Display the name of the column which is used
print("Image ID column:", image_column_id)

In [ ]:
# Find the file path for a given id 
def find_image_path(image_id):
    # Convert the id to a string, allowing it to be used as a dictionary key
    image_id = str(image_id)
    # Check whether the id exists 
    if image_id in image_lookup:
        # Return the path which is associaed with the id
        return image_lookup[image_id]
    # Add the ISIC prefix to the id
    prefixed = f"ISIC_{image_id}"
    # Assess whether the id, with a prefix, exists
    if prefixed in image_lookup:
        # Return the path which is associated with the id, with a prefix
        return image_lookup[prefixed]
    # If no file can be found, none can be returned
    return None
# Find the path for each id, storing them within a new column
df["filepath"] = df[image_column_id].apply(find_image_path)
# Count the number of images which do not have a matching path
missing = df["filepath"].isna().sum()
# Display the total number of labelled images from within the dataframe
print("Total labelled images:", len(df))
# Output the number of labelled images which are successfully matched to files
print("Images successfully matched:", len(df) - missing)
# Display the number of labelled images for which no matching file is found
print("Missing images:", missing)
# Check whether any images could not be matched
if missing > 0:
    # Display the first few rows of where the path is missing
    display(df[df["filepath"].isna()].head())

In [ ]:
# Remove rows where no path is found and reset the index of the dataframe
df = df.dropna(subset=["filepath"]).reset_index(drop=True)
# Display the number of images which remain
print("Final usable dataset:", len(df))

In [ ]:
# Split the dataset into a training and temporary set, having training as 80%
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED)
# Split the temporary set equally into validation and test sets, being each at 10%
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
# Reset the index of the training, validation, and test dataframes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
# Display the number of images within the training, validation, and test dataframes
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
# Introduce the class distribution within each of the dataframes
print("\nClass distribution:")
# Calculate and display the proportion of each class within the dataframes
print("Train:", train_df["label"].value_counts(normalize=True).sort_index().to_dict())
print("Validation:", val_df["label"].value_counts(normalize=True).sort_index().to_dict())
print("Test:", test_df["label"].value_counts(normalize=True).sort_index().to_dict())

In [ ]:
# Transform the images which are used for training
train_transform = transforms.Compose([
    # Resize the images
    transforms.Resize((256, 256)),
    # Randomly crop a 224 x 224 section of the image
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    # Randomly flip the image horizontally, with a 50% probability
    transforms.RandomHorizontalFlip(p=0.5),
    # Randomly flip the image vertically, with a 20% probability
    transforms.RandomVerticalFlip(p=0.2),
    # Randomly rotate the image in either direction, being up to 15 degrees
    transforms.RandomRotation(15),
    # Adjust the brightness, contrast, saturation, and hue, creating small variations in appearance
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    # Convert the image into a format which PyTorch can work with
    transforms.ToTensor(),
    # Normalise the value, using the mean and standard deviation
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Transform the images which are used for validation and testing
eval_transform = transforms.Compose([
    # Resize the images 
    transforms.Resize((224, 224)),
    # Convert the image into a format which PyTorch can work with
    transforms.ToTensor(),
    # Normalise the value, using the mean and standard deviation
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
# Form a dataset class, for each of the images and their labels
class SkinCancerDataset(Dataset):
    # Using the dataset and image transformations, set up the dataframe
    def __init__(self, dataframe, transform=None):
        # Store the dataframe and reset its index, ensuring it starts from 0
        self.dataframe = dataframe.reset_index(drop=True)
        # Store the transformations which are going to be applied to each image
        self.transform = transform

    # Return the total number of images from within the dataset
    def __len__(self):
        # Count the number of rows from within the dataset
        return len(self.dataframe)

    # Get one image and its corresponding label from the dataset, using its position
    def __getitem__(self, idx):
        # Obtain the row which corresponds to the images which is requested
        row = self.dataframe.iloc[idx]
        # Open the image, using its path, and convert it to RGB colour
        image = Image.open(row["filepath"]).convert("RGB")
        # Assess whether any transformations are applied
        if self.transform is not None:
            # Apply the transformations to the image 
            image = self.transform(image)
        # Convert the label into a PyTorch value, allowing it to be used by the model
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        # Return the image and its label 
        return image, label

In [ ]:
# Define the number of images which are processed at one time
BATCH_SIZE = 32
# Create the training dataset, applying the training transformations 
train_dataset = SkinCancerDataset(train_df, transform=train_transform)
# Create th validation and test datasets, applying the evaluation transformations
val_dataset = SkinCancerDataset(val_df, transform=eval_transform)
test_dataset = SkinCancerDataset(test_df, transform=eval_transform)
# Create a dataloader for the training dataset, mixing the order of the images
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
# Create a dataloader for the validation and test datasets, keeping them in order
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
# Display the number of batches within each dataset
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
class ResNet18Baseline(nn.Module):
    # Set-up the ResNet-18 model and its final output layer
    def __init__(self, num_classes=2):
        # Setup the PyTorch module 
        super().__init__()
        # Create a ResNet-18 model, without using pre-trained weights
        self.model = models.resnet18(weights=None)
        # Find the number of input features which are going into the final layer of the model
        num_features = (self.model.fc.in_features)
        # Replace the original final layer with a dropout, followed by a new linear layer which produces one output for each class
        self.model.fc = nn.Sequential(nn.Dropout(p=0.30), nn.Linear(num_features, num_classes))

    # Extract the learned features
    def extract_features(self, x):
        # Pass the image through the convolutional layers
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        # Continue through the remaining layers
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        # Reduce the spatial feature map to one value for each feature
        x = self.model.avgpool(x)
        # Flatten the features into one row for each image
        x = torch.flatten(x, 1)
        # Return the learned features
        return x

    # Define how an image is passed through the model, to produce a prediction
    def forward(self, x):
        # Pass the image through the feature extractor
        features = self.extract_features(x)
        # Pass the extracted features through the final classifier
        return self.model.fc(features)

In [ ]:
class ViTBaseClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        num_features = (self.model.heads.head.in_features)
        self.model.heads.head = nn.Linear(num_features, num_classes)

    def extract_features(self, x):
        x = self.model._process_input(x)
        batch_size = x.shape[0]
        cls_token = self.model.class_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = self.model.encoder(x)
        return x[:, 0]
        
    def forward(self, x):
        features = self.extract_features(x)
        return self.model.heads(features)

In [ ]:
# Obtain the output of a model, for a group of images
def get_logits(model, images):
    # Pass the images through the model
    output = model(images)
    # Return the output of the model
    return output

In [ ]:
def evaluate_model(model, loader, criterion, threshold=0.5):
    # Set the model so that it can evaluate images
    model.eval()
    # Keep track of the total loss, across each of the images
    running_loss = 0.0
    # Store the correct label for each image
    all_labels = []
    # Store the probabilities in which each model predicts
    all_probs = []
    # Prevent the model from calculating gradients
    with torch.no_grad():
        # Process the image in batches 
        for images, labels in loader:
            # Move each of the images, and their corresponding labels, to the device which is being used
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            # Allow the models to form a prediction on each of the images
            logits = get_logits(model, images)
            # Calculate the loss between the prediction and label 
            loss = criterion(logits, labels)
            # Add the loss from this batch to the overall batch
            running_loss += (loss.item() * images.size(0))
            # Convert the outputs of the model into probabilities, taking the probability of melanoma
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            # Store the probabilities and correct labels for the current batch
            all_probs.extend(probabilities.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    # Calculate the average loss across the whole dataset
    avg_loss = (running_loss / len(loader.dataset))
    # Convert the lables and probabilities, each into a seperate array
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    # Turn the probabilities into predicted classes, using the given threshold
    predictions = (all_probs >= threshold).astype(int)
    # Calculate the number of predictions which are correct, and how often melanoma is actually melanoma, calculating the number of melanoma cases which are correctly identified
    accuracy = accuracy_score(all_labels, predictions)
    precision = precision_score(all_labels, predictions, zero_division=0) 
    recall = recall_score(all_labels, predictions, zero_division=0)
    # Combine precision and recall into a single value, reflecting
    f1 = f1_score(all_labels, predictions, zero_division=0)
    # Calculate how well the predicted probabilities distinguish between the two classes
    auc = roc_auc_score(all_labels, all_probs)
    # Return all of the results
    return {"loss": avg_loss, "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "auc": auc, "labels": all_labels, "probabilities": all_probs, "predictions": predictions}

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, scheduler=None, epochs=30, patience=7, threshold=0.5, model_name="Model"):
    # Define the loss function which is used to measure how far the predictions of the model are from  the true labels
    criterion = nn.CrossEntropyLoss()
    # Set the intial best validation accuracy to negative infinity, ensuring the first epoch will always improve on it
    best_val_accuracy = -np.inf
    # Store a copy of the parameters of the model, before training, ensuring that there is a valid starting state to fall back on
    best_state = copy.deepcopy(model.state_dict())
    # Create lists to store the training and validation metrics which are recorded after each epoch 
    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": [], "val_precision": [], "val_recall": [], "val_f1": [], "val_auc": []}
    # Count how many consecutive epochs have passed without an improvement in validation accuracy
    epochs_without_improvement = 0
    # Repeat the training process for the specified number of epochs
    for epoch in range(epochs):
        # Put the model into training mode, ensuring layers which are specific to training behave appropriately
        model.train()
        # Initialise the total training loss for the current epoch, as well as the number of correctly classified images, and the total number of images which are used in training 
        running_loss = 0.0
        correct = 0
        total = 0
        # Process the training data, batch by batch
        for images, labels in train_loader:
            # Move the input images and their corresponding labels to the selected device, being the GPU
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            # Clear the gradients from the previous batch
            optimizer.zero_grad()
            # Produce the raw scores for the current batch
            logits = get_logits(model, images)
            # Calculate the loss by comparing the predicted scores with the true labels
            loss = criterion(logits, labels)
            # Calculate the gradients of the loss with respect to the parameters of the model
            loss.backward()
            # Update the parameters of the model, using the calculated gradients
            optimizer.step()
            #  Add the loss from the current batch to the running total, weighted by the number of images within the batch
            running_loss += (loss.item() * images.size(0))
            # Select the class with the highest predicted score for each image
            predictions = torch.argmax(logits, dim=1)
            # Count the number of predictions which match the true labels
            correct += (predictions == labels).sum().item()
            # Add the number of images from within the batch to the total number which is prcoessed
            total += labels.size(0)
        # Calculate the average training loss across the entire training set, and the proportion of images which are correctly classified
        train_loss = (running_loss / len(train_loader.dataset))
        train_accuracy = (correct / total)
        # Evaluate the trained model on the validation set, using the specified classification threshold
        val_metrics = evaluate_model(model, val_loader, criterion, threshold=threshold)
        # Add each of the resuts to history
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_f1"].append(val_metrics["f1"])
        history["val_auc"].append(val_metrics["auc"])
        # Display the results
        print(f"[{model_name}] " f"Epoch {epoch + 1}/{epochs} | " f"Train Loss: {train_loss:.4f} | " f"Train Acc: {train_accuracy:.4f} | " f"Val Loss: {val_metrics['loss']:.4f} | " f"Val Acc: {val_metrics['accuracy']:.4f} | " f"Val F1: {val_metrics['f1']:.4f} | " f"Val AUC: {val_metrics['auc']:.4f}")
        # Check whether there is a learning rate scheduler
        if scheduler is not None:
            # Update the learning rate, using the validation accuracy from the current epoch
            scheduler.step(val_metrics["accuracy"])
        # Check whether the current validation accuracy is better than the best seen so far
        if (val_metrics["accuracy"] > best_val_accuracy):
            # Update the best validation accuracy
            best_val_accuracy = (val_metrics["accuracy"])
            # Save a copy of the model parameters from the current best epoch
            best_state = copy.deepcopy(model.state_dict())
            # Reset the counter in observing an improvement
            epochs_without_improvement = 0
        else:
            # Increase the counter when validation accuracy does not improve
            epochs_without_improvement += 1
        # Stop training, when the validation accuracy has failed to improve for the specified value of patience
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch " f"{epoch + 1}.")
            break
    model.load_state_dict(best_state)
    print(f"\nBest validation accuracy " f"({model_name}): " f"{best_val_accuracy:.4f}")
    return model, history

In [ ]:
def find_best_threshold(labels, probabilities, metric="accuracy"):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.5
    best_score = -np.inf
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        if metric == "accuracy":
            score = accuracy_score(labels, predictions)
        elif metric == "f1":
            score = f1_score(labels, predictions, zero_division=0)
        else:
            raise ValueError("metric must be 'accuracy' or 'f1'")
        if score > best_score:
            best_score = score
            best_threshold = threshold
    return best_threshold, best_score

In [ ]:
baseline_model = ResNet18Baseline(num_classes=2).to(device)
baseline_optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=1e-4, weight_decay=1e-4)
baseline_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(baseline_optimizer, mode="max", factor=0.5, patience=3)
baseline_model, baseline_history = train_model(model=baseline_model, train_loader=train_loader, val_loader=val_loader, optimizer=baseline_optimizer, scheduler=baseline_scheduler, epochs=25, patience=7, threshold=0.5, model_name="ResNet-18")

In [ ]:
vit_model = ViTBaseClassifier(num_classes=2).to(device)
vit_optimizer = torch.optim.AdamW(vit_model.parameters(), lr=1e-4, weight_decay=0.05)
vit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(vit_optimizer, mode="max", factor=0.5, patience=3)
vit_model, vit_history = train_model(model=vit_model, train_loader=train_loader, val_loader=val_loader, optimizer=vit_optimizer, scheduler=vit_scheduler, epochs=20, patience=7, threshold=0.5, model_name="ViT-B/16")

In [ ]:
class CNNVisionTransformer(nn.Module):
    def __init__(self, resnet_model, vit_model, num_classes=2, dropout=0.30):
        super().__init__()
        # Store the models, ResNet-18 and ViT-B/16
        self.resnet = resnet_model
        self.vit = vit_model
        # Set the number of features in which each of the models produce
        cnn_dim = 512
        vit_dim = 768
        # Calculate the number of features which exist in combining the two models
        fusion_dim = (cnn_dim + vit_dim)
        # Define the layers which are used
        self.classifier = nn.Sequential(nn.Linear(fusion_dim, 512), nn.GELU(), nn.Dropout(dropout), nn.Linear(512, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        # Extract the visual features in which ResNet-18 and ViT-B/16 learn 
        cnn_features = self.resnet.extract_features(x)
        vit_features = self.vit.extract_features(x)
        # Join the features from the two models into a single representation
        fused_features = torch.cat([cnn_features, vit_features], dim=1)
        # Pass the combined representation through the classifier
        logits = self.classifier(fused_features)
        # Return the scores which are produced for each class
        return logits

In [ ]:
hybrid_model = CNNVisionTransformer(resnet_model=copy.deepcopy(baseline_model), vit_model=copy.deepcopy(vit_model), num_classes=2, dropout=0.30).to(device)

In [ ]:
hybrid_optimizer = torch.optim.AdamW([{"params": hybrid_model.resnet.parameters(), "lr": 1e-5}, {"params": hybrid_model.vit.parameters(), "lr": 1e-5}, {"params": hybrid_model.classifier.parameters(), "lr": 1e-4}], weight_decay=0.01)
hybrid_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(hybrid_optimizer, mode="max", factor=0.5, patience=3)
hybrid_model, hybrid_history = train_model(model=hybrid_model, train_loader=train_loader, val_loader=val_loader, optimizer=hybrid_optimizer, scheduler=hybrid_scheduler, epochs=10, patience=7, threshold=0.5, model_name="CNN-ViT Hybrid")

In [ ]:
resnet_target = baseline_model.model.layer4[-1]
resnet_target._forward_hooks.clear()
resnet_target._backward_hooks.clear()
vit_target = vit_model.model.encoder.layers[-1].ln_1
vit_target._forward_hooks.clear()
vit_target._backward_hooks.clear()
hybrid_resnet_target = hybrid_model.resnet.model.layer4[-1]
hybrid_vit_target = hybrid_model.vit.model.encoder.layers[-1].ln_1
hybrid_resnet_target._forward_hooks.clear()
hybrid_resnet_target._backward_hooks.clear()
hybrid_vit_target._forward_hooks.clear()
hybrid_vit_target._backward_hooks.clear()
baseline_model.eval()
vit_model.eval()
hybrid_model.eval()
melanoma_indices = test_df.index[test_df["MEL"].astype(int) == 1].tolist()
selected_images = []
for idx in melanoma_indices:
    image_path = test_df.loc[idx, "filepath"]
    image = Image.open(image_path).convert("RGB")
    candidate_tensor = (eval_transform(image).unsqueeze(0).to(device))
    with torch.no_grad():
        resnet_logits = baseline_model(candidate_tensor)
        resnet_probability = torch.softmax(resnet_logits, dim=1)[0, 1].item()
        vit_logits = vit_model(candidate_tensor)
        vit_probability = torch.softmax(vit_logits, dim=1)[0, 1].item()
        hybrid_logits = hybrid_model(candidate_tensor)
        hybrid_probability = torch.softmax(hybrid_logits, dim=1)[0, 1].item()
    if (resnet_probability >= 0.5 and vit_probability >= 0.5 and hybrid_probability >= 0.5):
        selected_images.append({
            "idx": idx,
            "image_path": image_path,
            "image": image,
            "resnet_probability": resnet_probability,
            "vit_probability": vit_probability,
            "hybrid_probability": hybrid_probability
        })

    if len(selected_images) == 3:
        break

if len(selected_images) < 3:
    raise ValueError(f"Only {len(selected_images)} melanoma images " f"were found that all three models classify as melanoma.")


print(f"Selected {len(selected_images)} melanoma images.")
for selected in selected_images:
    print(
        f"Index {selected['idx']} | "
        f"ResNet P = {selected['resnet_probability']:.3f} | "
        f"ViT P = {selected['vit_probability']:.3f} | "
        f"Hybrid P = {selected['hybrid_probability']:.3f}"
    )


def calculate_resnet_gradcam(input_tensor):
    baseline_model.eval()
    baseline_model.zero_grad(set_to_none=True)
    storage = {"activation": None, "gradient": None}

    def forward_hook(module, inputs, output):
        storage["activation"] = output

    def backward_hook(module, grad_input, grad_output):
        storage["gradient"] = grad_output[0]

    forward_handle = (resnet_target.register_forward_hook(forward_hook))
    backward_handle = (resnet_target.register_full_backward_hook(backward_hook))

    try:
        with torch.enable_grad():
            logits = baseline_model(input_tensor)
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            melanoma_logit = logits[:, 1]
            melanoma_logit.backward()

        activation = (storage["activation"].detach())
        gradient = (storage["gradient"].detach())
        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activation).sum(dim=1)
        cam = F.relu(cam)
        cam = F.interpolate(cam.unsqueeze(1), size=(224, 224), mode="bilinear", align_corners=False).squeeze()
        cam = (cam.cpu().numpy())
        if cam.max() > 0:
            cam = (cam / cam.max())
        return cam, probability
    finally:
        forward_handle.remove()
        backward_handle.remove()

def calculate_vit_gradcam(input_tensor):
    vit_model.eval()
    vit_model.zero_grad(set_to_none=True)
    storage = {"activation": None, "gradient": None}

    def forward_hook(module, inputs, output):
        storage["activation"] = output

    def backward_hook(module, grad_input, grad_output):
        storage["gradient"] = grad_output[0]

    forward_handle = (vit_target.register_forward_hook(forward_hook))
    backward_handle = (vit_target.register_full_backward_hook(backward_hook))

    try:
        with torch.enable_grad():
            logits = vit_model(input_tensor)
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            melanoma_logit = logits[:, 1]
            melanoma_logit.backward()
            
        activation = (storage["activation"].detach())
        gradient = (storage["gradient"].detach())
        activation = activation[:, 1:, :]
        gradient = gradient[:, 1:, :]
        weights = gradient.mean(dim=1, keepdim=True)
        cam = (weights * activation).sum(dim=2)
        cam = F.relu(cam)
        cam = cam.reshape(1, 1, 14, 14)
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False).squeeze()
        cam = (cam.cpu().numpy())
        if cam.max() > 0:
            cam = (cam / cam.max())
        return cam, probability

    finally:
        forward_handle.remove()
        backward_handle.remove()

def calculate_hybrid_gradcam(input_tensor):
    hybrid_model.eval()
    hybrid_model.zero_grad(set_to_none=True)
    storage = {"resnet_activation": None, "resnet_gradient": None, "vit_activation": None, "vit_gradient": None}

    def resnet_forward_hook(module, inputs, output):
        storage["resnet_activation"] = output

    def vit_forward_hook(module, inputs, output):
        storage["vit_activation"] = output

    def resnet_backward_hook(module, grad_input, grad_output):
        storage["resnet_gradient"] = grad_output[0]


    def vit_backward_hook(module, grad_input, grad_output):
        storage["vit_gradient"] = grad_output[0]

    resnet_forward_handle = (hybrid_resnet_target.register_forward_hook(resnet_forward_hook))
    resnet_backward_handle = (hybrid_resnet_target.register_full_backward_hook(resnet_backward_hook))
    vit_forward_handle = (hybrid_vit_target.register_forward_hook(vit_forward_hook))
    vit_backward_handle = (hybrid_vit_target.register_full_backward_hook(vit_backward_hook))
    try:
        with torch.enable_grad():
            logits = hybrid_model(input_tensor)
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            melanoma_logit = logits[:, 1]
            melanoma_logit.backward()
        resnet_activation = (storage["resnet_activation"].detach())
        resnet_gradient = (storage["resnet_gradient"].detach())
        vit_activation = (storage["vit_activation"].detach())
        vit_gradient = (storage["vit_gradient"].detach())
        resnet_weights = resnet_gradient.mean(dim=(2, 3), keepdim=True)
        hybrid_resnet_cam = (resnet_weights * resnet_activation).sum(dim=1)
        hybrid_resnet_cam = F.relu(hybrid_resnet_cam)
        hybrid_resnet_cam = F.interpolate(hybrid_resnet_cam.unsqueeze(1), size=(224, 224), mode="bilinear", align_corners=False).squeeze()
        hybrid_resnet_cam = (hybrid_resnet_cam.cpu().numpy())
        if hybrid_resnet_cam.max() > 0:
            hybrid_resnet_cam = (hybrid_resnet_cam / hybrid_resnet_cam.max())
        vit_activation = vit_activation[:, 1:, :]
        vit_gradient = vit_gradient[:, 1:, :]
        vit_weights = vit_gradient.mean(dim=1, keepdim=True)
        hybrid_vit_cam = (vit_weights * vit_activation).sum(dim=2)
        hybrid_vit_cam = F.relu(hybrid_vit_cam)
        hybrid_vit_cam = hybrid_vit_cam.reshape(1, 1, 14, 14)
        hybrid_vit_cam = F.interpolate(hybrid_vit_cam, size=(224, 224), mode="bilinear", align_corners=False).squeeze()
        hybrid_vit_cam = (hybrid_vit_cam.cpu().numpy())
        if hybrid_vit_cam.max() > 0:
            hybrid_vit_cam = (hybrid_vit_cam / hybrid_vit_cam.max())
        hybrid_cam = (0.5 * hybrid_resnet_cam + 0.5 * hybrid_vit_cam)
        hybrid_cam = np.maximum(hybrid_cam, 0)
        if hybrid_cam.max() > 0:
            hybrid_cam = (hybrid_cam / hybrid_cam.max())
        return (hybrid_cam, probability)
    finally:
        resnet_forward_handle.remove()
        resnet_backward_handle.remove()
        vit_forward_handle.remove()
        vit_backward_handle.remove()
fig, axes = plt.subplots(len(selected_images), 4, figsize=(18, 5 * len(selected_images)))
if len(selected_images) == 1:
    axes = axes.reshape(1, 4)
for row, selected in enumerate(selected_images):
    image = selected["image"]
    input_tensor = (eval_transform(image).unsqueeze(0).to(device).requires_grad_(True))
    resnet_cam, resnet_probability = (calculate_resnet_gradcam(input_tensor))
    vit_cam, vit_probability = (calculate_vit_gradcam(input_tensor))
    hybrid_cam, hybrid_probability = (calculate_hybrid_gradcam(input_tensor))
    display_image = image.resize((224, 224))
    axes[row, 0].imshow(display_image)
    axes[row, 0].set_title("Original Image")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(display_image)
    axes[row, 1].imshow(resnet_cam, cmap="jet", alpha=0.50)
    axes[row, 1].set_title(f"ResNet-18 Baseline\n" f"Melanoma P = {resnet_probability:.3f}")
    axes[row, 1].axis("off")
    axes[row, 2].imshow(display_image)
    axes[row, 2].imshow(vit_cam, cmap="jet", alpha=0.50)
    axes[row, 2].set_title(f"ViT-B/16 Baseline\n" f"Melanoma P = {vit_probability:.3f}")
    axes[row, 2].axis("off")
    axes[row, 3].imshow(display_image)
    axes[row, 3].imshow(hybrid_cam, cmap="jet", alpha=0.50)
    axes[row, 3].set_title(f"CNN-ViT Hybrid\n" f"Melanoma P = {hybrid_probability:.3f}")
    axes[row, 3].axis("off")
fig.suptitle("Grad-CAM Comparison for Melanoma Classification", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].imshow(display_image)
axes[0].imshow(resnet_cam, cmap="jet", alpha=0.45)
axes[0].set_title(
    f"ResNet-18\n"
    f"Melanoma P = {resnet_probability:.3f}"
)

axes[1].imshow(display_image)
axes[1].imshow(vit_cam, cmap="jet", alpha=0.45)
axes[1].set_title(
    f"ViT-B/16\n"
    f"Melanoma P = {vit_probability:.3f}"
)

axes[2].imshow(display_image)
axes[2].imshow(hybrid_cam, cmap="jet", alpha=0.45)
axes[2].set_title(
    f"CNN-ViT Hybrid\n"
    f"Melanoma P = {hybrid_probability:.3f}\n"
    f"ResNet: {cnn_weight * 100:.1f}% | "
    f"ViT: {vit_weight * 100:.1f}%"
)
for ax in axes:
    ax.axis("off")

fig.suptitle("Diagnosis: Melanoma", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
def get_test_predictions(model, test_loader):
    model.eval()
    all_labels = []
    all_predictions = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            output = model(images)
            if isinstance(output, tuple):
                output = output[0]
            probabilities = torch.softmax(output, dim=1)
            predictions = torch.argmax(probabilities, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
    return np.array(all_labels), np.array(all_predictions)
resnet_labels, resnet_predictions = get_test_predictions(baseline_model, test_loader)
vit_labels, vit_predictions = get_test_predictions(vit_model, test_loader)
hybrid_labels, hybrid_predictions = get_test_predictions(hybrid_model, test_loader)
resnet_cm = confusion_matrix(resnet_labels, resnet_predictions, labels=[0, 1])
vit_cm = confusion_matrix(vit_labels, vit_predictions, labels=[0, 1])
hybrid_cm = confusion_matrix(hybrid_labels, hybrid_predictions, labels=[0, 1])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cms = [(resnet_cm, "ResNet-18"), (vit_cm, "ViT-B/16"), (hybrid_cm, "CNN-ViT Hybrid")]
for ax, (cm, title) in zip(axes, cms):
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Melanoma", "Melanoma"])
    ax.set_yticklabels(["Non-Melanoma", "Melanoma"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

In [ ]:
def clean_image(image):
    return image

def blur_image(image):
    return image.filter(ImageFilter.GaussianBlur(radius=4))
    
def low_light_image(image):
    return ImageEnhance.Brightness(image).enhance(0.5)

def high_light_image(image):
    return ImageEnhance.Brightness(image).enhance(1.5)

class CorruptedSkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None, corruption=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.corruption = corruption

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = self.dataframe.loc[idx, "filepath"]
        label = int(self.dataframe.loc[idx, "label"])
        image = Image.open(image_path).convert("RGB")
        if self.corruption is not None:
            image = self.corruption(image)
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label

In [ ]:
corruption_functions = {"Clean": clean_image, "Blur": blur_image, "Low Lighting": low_light_image, "High Lighting": high_light_image}
corrupted_loaders = {}
for name, corruption in corruption_functions.items():
    dataset = CorruptedSkinCancerDataset(dataframe=test_df, transform=eval_transform, corruption=corruption)
    corrupted_loaders[name] = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
def evaluate_corruption(model, loader):
    model.eval()
    all_labels = []
    all_probabilities = []
    all_predictions = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            output = model(images)
            if isinstance(output, tuple):
                output = output[0]
            probabilities = torch.softmax(output, dim=1)[:, 1]
            predictions = (probabilities >= 0.5).long()
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)
    all_predictions = np.array(all_predictions)
    return {
        "Accuracy": accuracy_score( all_labels, all_predictions),
        "Precision": precision_score(all_labels, all_predictions, zero_division=0),
        "Recall": recall_score(all_labels, all_predictions, zero_division=0),
        "F1": f1_score(all_labels, all_predictions, zero_division=0),
        "AUC": roc_auc_score(all_labels, all_probabilities)
    }

In [ ]:
models_to_test = {"ResNet-18": baseline_model, "ViT-B/16": vit_model, "CNN-ViT Hybrid": hybrid_model}
results = []
for corruption_name, loader in corrupted_loaders.items():
    for model_name, model in models_to_test.items():
        metrics = evaluate_corruption(model, loader)
        results.append({
            "Condition": corruption_name,
            "Model": model_name,
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1": metrics["F1"],
            "AUC": metrics["AUC"]
        })
corruption_results = pd.DataFrame(results)
corruption_results

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    range(1, len(baseline_history["train_accuracy"]) + 1),
    baseline_history["train_accuracy"],
    marker="o",
    label="ResNet-18"
)

plt.plot(
    range(1, len(vit_history["train_accuracy"]) + 1),
    vit_history["train_accuracy"],
    marker="o",
    label="ViT-B/16"
)

plt.plot(
    range(1, len(hybrid_history["train_accuracy"]) + 1),
    hybrid_history["train_accuracy"],
    marker="o",
    label="CNN-ViT Hybrid"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 6))

plt.plot(
    range(1, len(baseline_history["val_accuracy"]) + 1),
    baseline_history["val_accuracy"],
    marker="o",
    label="ResNet-18"
)

plt.plot(
    range(1, len(vit_history["val_accuracy"]) + 1),
    vit_history["val_accuracy"],
    marker="o",
    label="ViT-B/16"
)

plt.plot(
    range(1, len(hybrid_history["val_accuracy"]) + 1),
    hybrid_history["val_accuracy"],
    marker="o",
    label="CNN-ViT Hybrid"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()